# Try it: agri-llm, Qwen2.5-7B + LoRA adapter

> **Verified on Kaggle T4 on 2026-11-15; scores matched the committed R4 results.**

> **Research / portfolio model. NOT for real pesticide advice.** Always follow the
> product label and a local agriculture officer.

For three benchmark questions, this notebook:
1. builds the same fact-sheet prompt the R4 evaluation used (`src/retrieval.py`);
2. generates an answer with the adapter (greedy, `max_new_tokens=3072`,
   `repetition_penalty=1.05`);
3. runs the deterministic guard (`src/guard.py`);
4. scores the answer, with and without the guard, using the project's verifier
   (`src/verify.py`).

**Runtime:** Kaggle or Colab, **T4 GPU, internet on**.

- Model: https://huggingface.co/Janhvi1008/agri-llm-qwen2.5-7b-lora
- Code: https://github.com/janhvipatil1008-netizen/Finetuning-agrillm

## 1. Install Unsloth and clone the repository

In [ ]:
!pip install -q unsloth
!git clone -q --depth 1 https://github.com/janhvipatil1008-netizen/Finetuning-agrillm.git agri-llm
%cd agri-llm

## 2. Rebuild `label_db.parquet`

The repository ignores `*.parquet`, so a fresh clone has only `label_db.csv`.

`verify.py` refuses the CSV on purpose. `phi_days` degrades from nullable `Int64` to
`float64` through CSV, which blurs "PHI is 0 days" and "PHI unknown".

This cell rebuilds the parquet from the CSV with the original's exact dtypes. The
recipe was checked against the real file and matches it value for value and dtype for
dtype.

In [ ]:
from pathlib import Path
import pandas as pd

# Non-text columns of the real parquet. Every other column is text, with "" (never None) for an empty cell.
LABEL_DB_DTYPES = {
    "crop_multi_crop_split": "bool", "flag_pest_bled": "bool", "phi_not_applicable": "bool",
    "phi_days": "Int64",
    "dose_ai_value_min": "float64", "dose_ai_value_max": "float64",
    "dose_ai_per_acre_min": "float64", "dose_ai_per_acre_max": "float64",
    "dose_formulation_value_min": "float64", "dose_formulation_value_max": "float64",
    "dose_formulation_per_acre_min": "float64", "dose_formulation_per_acre_max": "float64",
}


def rebuild_label_db(csv_path, out_path):
    csv = pd.read_csv(csv_path, keep_default_na=False, na_values=[""], dtype=str)
    df = pd.DataFrame(index=csv.index)
    for col in csv.columns:
        kind, s = LABEL_DB_DTYPES.get(col), csv[col]
        if kind == "Int64":
            df[col] = pd.to_numeric(s).astype("Int64")
        elif kind == "float64":
            df[col] = pd.to_numeric(s).astype("float64")
        elif kind == "bool":
            assert s.isin(["True", "False"]).all(), f"{col}: not a clean True/False column"
            df[col] = s.map({"True": True, "False": False}).astype(bool)
        else:
            df[col] = s.fillna("").astype(object)
    df.to_parquet(out_path)
    return df


LABEL_DB = Path("data/final/label_db.parquet")
if not LABEL_DB.exists():
    df = rebuild_label_db("data/final/label_db.csv", LABEL_DB)
    print(f"rebuilt {LABEL_DB}: {len(df)} rows, {int(df['phi_days'].isna().sum())} unknown PHI, "
          f"{int((df['phi_days'] == 0).sum())} zero-day PHI")

## 3. Load the label data, verifier and guard (CPU)

In [ ]:
import json
import sys

sys.path[:0] = ["src", "tools"]
import verify
from guard import guard_detailed
from phase10_stepB_audit import ctx_for          # the scoring convention tools/score.py uses
from retrieval import build_rag_user_message, miss_reason_for, resolve_key, retrieve

res = verify.VerifyResources.load()
SYSTEM = Path("src/system_prompt.txt").read_text(encoding="utf-8")   # the frozen prompt the adapter was trained with
bench = {r["item_id"]: r for r in map(json.loads, open("data/final/bench.jsonl", encoding="utf-8"))}
print(f"label_db rows: {len(res.label_db.rows)}, benchmark items: {len(bench)}")

## 4. Three benchmark questions and their fact-sheet prompts

| item | what it shows |
|---|---|
| `S1_140` | a plain dose answer (cotton wilt) |
| `S1_439` | an answer with an unknown pre-harvest interval, which the guard escalates (cotton thrips) |
| `B4b_7` | no crop resolved: "angur" (grape) is not in the crop mapper |

The prompt is rebuilt here with `src/retrieval.py`: the Phase 12 Arm B live retrieval,
capped at 15 rows. It is then asserted equal to the committed R4 prompt.

In [ ]:
ITEMS = {
    "S1_140": "plain dose answer: cotton wilt",
    "S1_439": "unknown-PHI answer the guard escalates: cotton thrips",
    "B4b_7": "no crop resolved: 'angur' (grape) is not in the crop mapper",
}


def fact_sheet_prompt(query):
    """The R4 user message: key from the query alone, label rows as a fact sheet."""
    crop, pest, method = resolve_key(query, res)
    ret = retrieve(crop, pest, res, method)
    user = build_rag_user_message(query, crop, ret.rows, resolved_pest=None,
                                  miss_reason=miss_reason_for(crop, pest),
                                  rows_available=ret.rows_available)
    return user, (crop, pest, method)


committed = {r["item_id"]: r for r in map(json.loads, open("data/final/prompts/prompts_R4.jsonl", encoding="utf-8"))}
prompts = {}
for iid, what in ITEMS.items():
    user, key = fact_sheet_prompt(bench[iid]["query_text"])
    assert user == committed[iid]["user"], f"{iid}: prompt differs from the committed R4 prompt"
    prompts[iid] = user
    print(f"{iid} ({what})\n  query: {bench[iid]['query_text']}\n  key (crop, pest, method): {key}\n")
print(prompts["S1_140"][:1200])

### What the recorded R4 run did on these items

On a different GPU or library version, greedy decoding can change the text without
changing the score. Phase 14 reran the same prompts and compared the 425 answers that
had finished: 105 changed in text, but only 4 of 424 changed in score
(`reports/phase14_clean_eval.md`, §7).

In [ ]:
rec = {r["item_id"]: r for r in json.load(open("data/final/eval/scores_R4_sft_rag.json", encoding="utf-8"))["items"]}
rec_g = {r["item_id"]: r for r in json.load(open("data/final/eval/scores_R4_sft_rag_guarded.json", encoding="utf-8"))["items"]}
acts = {r["item_id"]: r["guard_actions"] for r in map(json.loads, open("data/final/eval/guarded/R4_sft_rag_responses.jsonl", encoding="utf-8"))}
for iid in ITEMS:
    print(f"{iid}: recorded score {rec[iid]['score']:.3f} -> after guard {rec_g[iid]['score']:.3f}; "
          f"guard actions {[a['action'] for a in acts[iid]] or 'none'}")

## 5. Load the base model and the adapter (GPU)

In [ ]:
from unsloth import FastLanguageModel             # import unsloth before transformers / peft
import torch
from peft import PeftModel

BASE = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
ADAPTER = "Janhvi1008/agri-llm-qwen2.5-7b-lora"

model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=5911,
                                                     dtype=None, load_in_4bit=True)
model = PeftModel.from_pretrained(model, ADAPTER)
lora_b = sum(p.detach().float().abs().sum().item() for n, p in model.named_parameters() if "lora_B" in n)
assert lora_b > 0, "adapter weights did not load (lora_B is all zero)"
print(f"sum |lora_B| = {lora_b:.4g}")
FastLanguageModel.for_inference(model)

## 6. Generate, guard and score

In [ ]:
GEN = dict(max_new_tokens=3072, do_sample=False, num_beams=1, repetition_penalty=1.05,
           temperature=None, top_p=None, top_k=None, use_cache=True)   # as tools/generate.py ran R4
PAD = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id


def generate(system, user):
    text = tokenizer.apply_chat_template([{"role": "system", "content": system},
                                          {"role": "user", "content": user}],
                                         tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        out = model.generate(**enc, pad_token_id=PAD, **GEN)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
def score(answer, item):
    """verify() in score mode, against the item's gold label (tools/score.py's convention)."""
    r = verify.verify(answer, ctx_for(verify, res, item), "score")
    if r.excluded:
        return f"excluded ({r.exclusion_reason[:80]})"
    failed = [g for g, ok in r.gates.items() if not ok]
    return f"{r.total:.3f}" + (f" (failed gates: {', '.join(failed)})" if failed else "")


def show(answer):
    try:
        return json.dumps(json.loads(answer), indent=1, ensure_ascii=False)
    except json.JSONDecodeError:
        return answer


for iid, what in ITEMS.items():
    item = bench[iid]
    answer = generate(SYSTEM, prompts[iid])
    g = guard_detailed(answer, item["query_text"], res)
    print("=" * 100)
    print(f"{iid}: {what}\nQ: {item['query_text']}\n")
    print(show(answer)[:4000])
    print(f"\nguard key: crop={g.crop}, pest={g.pest} | actions: {[a['action'] for a in g.actions] or 'none'} "
          f"| changed: {g.changed}")
    print(f"verify() score: model alone {score(answer, item)} | with guard {score(g.output, item)}")